# 🌊 WaveTrader — Full Pipeline on Google Colab
### Clone → Preprocess JForex Data → Train FluxSignal + WaveTraderMTF → Save to Drive → Validate

**Runtime:** GPU (T4 or better) — *Runtime → Change runtime type → GPU*

**Before running:**
1. Upload your JForex CSV files to Google Drive under `MyDrive/phase_lm/data/`
2. Make sure your repo is on GitHub (or you'll upload manually in Section 2)
3. Enable GPU runtime

---
| Step | What happens |
|------|-------------|
| §1 | Mount Google Drive, create workspace dirs |
| §2 | Clone / upload repo code |
| §3 | Install dependencies |
| §4 | Explore raw data |
| §5 | Preprocess: 1m→15m resample, mid-price, filters, Parquet output |
| §6 | Define models (FluxSignal single-TF + WaveTraderMTF) |
| §7 | Configure hyperparameters |
| §8 | Train with progress curves |
| §9 | Save checkpoints + config to Google Drive |
| §10 | Reload model from Drive — verify weight integrity |
| §11 | Validation: accuracy, per-class F1, backtest on test holdout |

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os, pathlib

drive.mount("/content/drive")

# ── Workspace layout on Google Drive ────────────────────────────────────────
DRIVE_ROOT   = pathlib.Path("/content/drive/MyDrive/phase_lm")
DRIVE_DATA   = DRIVE_ROOT / "data"          # raw JForex CSVs live here
DRIVE_CKPT   = DRIVE_ROOT / "checkpoints"   # model saves
DRIVE_PROC   = DRIVE_ROOT / "processed_data"# preprocessed Parquet cache

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_PROC]:
    d.mkdir(parents=True, exist_ok=True)

# ── Colab-local workspace (fast NVMe on Colab) ───────────────────────────────
LOCAL_ROOT = pathlib.Path("/content/phase_lm")
LOCAL_DATA = LOCAL_ROOT / "data"
LOCAL_PROC = LOCAL_ROOT / "processed_data"

print("Google Drive mounted.")
print(f"  Drive checkpoints : {DRIVE_CKPT}")
print(f"  Drive data        : {DRIVE_DATA}")
print()

# List what's already on Drive
csv_files = list(DRIVE_DATA.glob("*.csv"))
print(f"CSV files found in Drive data dir: {len(csv_files)}")
for f in sorted(csv_files)[:10]:
    print(f"  {f.name}")
if len(csv_files) > 10:
    print(f"  … and {len(csv_files)-10} more")

## 2. Clone the Repository

Clone the `wavetrader` repository from GitHub.

In [ ]:
import shutil, subprocess, sys

# ── Configuration ─────────────────────────────────────────────────────────────
GITHUB_URL = "https://github.com/Unseengap/wavetrader.git"

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def _run(cmd: str) -> None:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)

print("Cloning from GitHub …")
_run(f"git clone {GITHUB_URL} {LOCAL_ROOT}")

# Add repo to Python path
sys.path.insert(0, str(LOCAL_ROOT))
print(f"\nRepo root: {LOCAL_ROOT}")

# Symlink data: point local data/ at the Drive copy (avoids duplicating ~30 GB)
local_data_link = LOCAL_ROOT / "data"
if not local_data_link.exists():
    local_data_link.symlink_to(DRIVE_DATA)
    print(f"Symlinked {local_data_link} → {DRIVE_DATA}")

print("\nDirectory structure:")
_run(f"ls -la {LOCAL_ROOT}")

## 3. Install Dependencies

In [ ]:
# Colab already ships torch, numpy, pandas, matplotlib.
# We only need pyarrow (Parquet) and (optionally) pytz.
!pip install -q pyarrow fastparquet

# ── Verify GPU ────────────────────────────────────────────────────────────────
import torch, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import os, pathlib, json, hashlib, warnings
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")
if device.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"NumPy    : {np.__version__}")
print(f"Pandas   : {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

## 4. Load and Explore the Dataset

Quick sanity-check on the raw JForex CSVs before preprocessing.

In [ ]:
PAIRS   = ["GBPJPY", "EURJPY", "GBPUSD"]  # Temporarily excluded USDJPY
TF_MAP  = {"1m": "1 Min", "1h": "Hourly", "4h": "4 Hours", "1d": "Daily"}
PIP_SIZE = {"GBPJPY": 0.01, "USDJPY": 0.01, "EURJPY": 0.01, "GBPUSD": 0.0001}

def _probe_csv(pair: str, tf_key: str, side: str) -> dict:
    frag = TF_MAP[tf_key]
    candidates = [
        f for f in DRIVE_DATA.glob(f"{pair}_{frag}_{side}_*.csv")
        if "(1)" not in f.name
    ]
    if not candidates:
        return {"pair": pair, "tf": tf_key, "side": side, "rows": 0, "found": False}
    p = sorted(candidates)[0]
    df = pd.read_csv(p, nrows=5, skipinitialspace=True)
    n  = sum(1 for _ in open(p)) - 1   # fast line count
    return {
        "pair": pair, "tf": tf_key, "side": side,
        "file": p.name, "rows": n, "found": True,
        "columns": list(df.columns),
        "sample_ts": df.iloc[0, 0] if len(df) else None,
    }

rows = []
for pair in PAIRS:
    for tf in TF_MAP:
        rows.append(_probe_csv(pair, tf, "Bid"))

summary = pd.DataFrame(rows)[["pair","tf","side","rows","found","file"]]
missing = summary[~summary["found"]]
if len(missing):
    print("⚠ MISSING FILES:")
    print(missing[["pair","tf","side"]].to_string(index=False))
else:
    print("✓ All 12 Bid files found.")

print("\nRow counts (Bid files):")
pivot = summary.pivot(index="pair", columns="tf", values="rows")
print(pivot.to_string())

# ── Visualise one pair's daily close series ───────────────────────────────────
sample_path = sorted(DRIVE_DATA.glob("GBPJPY_Daily_Bid_*.csv")
                     )[0] if list(DRIVE_DATA.glob("GBPJPY_Daily_Bid_*.csv")) else None
if sample_path:
    raw = pd.read_csv(sample_path, skipinitialspace=True,
                      parse_dates=["Time (EET)"])
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    axes[0].plot(raw["Time (EET)"], raw["Close"], lw=0.8, color="#2196F3")
    axes[0].set_title("GBPJPY Daily Bid Close  (raw EET timestamps)")
    axes[0].set_ylabel("Price")
    axes[1].hist(raw["Close"].pct_change().dropna() * 100,
                 bins=100, color="#4CAF50", edgecolor="none")
    axes[1].set_title("Daily returns distribution (%)")
    axes[1].set_xlabel("% change")
    plt.tight_layout()
    plt.show()
    print(f"\nDate range : {raw['Time (EET)'].min()} → {raw['Time (EET)'].max()}")
    print(f"Close stats:\n{raw['Close'].describe().round(4).to_string()}")

In [ ]:
import os, pathlib

# ── Save the Deployment & Warm-up documentation to Google Drive ──────────────
DOCS_DIR = DRIVE_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

doc1_content = """# WaveTrader Live Deployment Guide

## 1. Infrastructure Architecture
**Recommended Stack:** VPS + Containerization
*   **Primary:** Hetzner Cloud (CPX31) or DigitalOcean
*   **Specs:** 4 vCPU, 8GB RAM, 160GB NVMe SSD
*   **OS:** Ubuntu 22.04 LTS

**Why Not Heroku:** Dyno sleeping ruins the 25-hour temporal memory state; ephemeral storages wipe Resonance Buffers.

## 2. Docker Setup
`Dockerfile` (production):
```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
# Pin PyTorch version to avoid breaking changes in future rebuilds
RUN pip install torch==2.2.2+cpu --index-url https://download.pytorch.org/whl/cpu \\
    && pip install -r requirements.txt
COPY wavetrader/ ./wavetrader/
COPY config/ ./config/
VOLUME ["/data"]  # Persistent state storage
CMD ["python", "-m", "wavetrader.streaming"]
```

## 3. State Persistence Strategy (Critical)
**The Problem:**
If the process restarts cold, it needs 25 hours to "catch up" before producing valid signals.
*   **CWC (Causal Wave Chainer) State:** Holds 100-bar hidden state (~25 hours of memory) and must be saved to persistent disk.
*   **Resonance Buffer:** Contains "memories" of recent high-volatility events, typically stored in Redis.

**Solution: Continuous Checkpointing** (Every 100 bars)
Do not rely on Redis for the main neural network state. Use explicit `torch.save` to dump the exact structure directly to your persistent `/data/checkpoints` volume.
"""

doc2_content = """# WaveTrader Training Gap Management & Live Warm-Up Protocol

## 1. The Temporal Memory Problem
WaveTrader is stateful.
* **CWC Hidden State:** Holds 100-bar rolling window (25 hours).
* **Resonance Buffer:** Holds recent experiences.
Starting cold = invalid signals.

## 2. The 100-Bar Rule
The 15-minute lookback is the bottleneck. You must pipe at minimum **100 historical bars (25 continuous hours)** of recent API data through the system *before* going active with live WebSocket data.

## 3. Sub-Bar Interruptions (WebSocket Drops)
*   **The Issue:** High volatility triggers brief WebSocket lag/reconnects (10-30 seconds). An incomplete OHLC tick stream can distort the 15-minute bar.
*   **Action Logic:**
    1.  Hold partial bar in-memory.
    2.  On reconnect, perform a rapid REST check against the current 15m candle.
    3.  If `open` or `high`/`low` differs significantly, fast-fetch the entire candle from REST to patch the missing sub-bar ticks. Do **not** process a partial block as the final candle close.

## 4. Timezone/DST Transitions
*   **The Issue:** Handling DST conversions introduces length offsets during missing gaps.
*   **Action Logic:**
    1.  All data is standard UTC (conversion logic via `zoneinfo` explicitly shifts EET timestamps to naive UTC natively).
    2.  Instead of checking specific offset dates, simply verify that your 15m resampling logic **did not output any 30-minute gaps** when reconstructing bars near missing/outage days.

**Golden Rule:** If the model has been asleep for more than 24 hours, it must dream (backfill) before it can trade.
"""

with open(DOCS_DIR / "deployment_guide.md", "w") as f:
    f.write(doc1_content)

with open(DOCS_DIR / "warmup_and_backfill_protocol.md", "w") as f:
    f.write(doc2_content)

print(f"✓ Saved Deployment and Warm-up docs to your Google Drive: {DOCS_DIR}")


## 5. Preprocess and Transform the Data

Runs the full preprocessing pipeline from `scripts/preprocess_data.py`:
- Parses JForex EET timestamps → UTC (DST-correct via `zoneinfo`)
- Calculates mid-prices `(bid + ask) / 2` and spread in pips
- Resamples 1m → 15m
- Applies quality filters (flash crash, spread threshold, NaN)
- Adds RSI-14, ATR-14, log-returns, session flags
- Saves `{pair}_{tf}.parquet` per split to Drive for reuse

In [ ]:
import importlib, sys as _sys

# ── Route outputs to Drive so we don't rerun multi-hour preprocessing ─────────
FORCE_REPROCESS = True   # set True to redo from scratch

# Inject the repo into the preprocessing script's expected paths
_preprocess_spec = importlib.util.spec_from_file_location(
    "preprocess_data",
    str(LOCAL_ROOT / "scripts" / "preprocess_data.py"),
)
_preprocess_mod = importlib.util.module_from_spec(_preprocess_spec)

# Monkey-patch path constants before executing the module
# import preprocess_data as _pd_stub  # will fail — we need to set paths first
# ── Instead: run the script directly with env overrides ──────────────────────
import os
os.chdir(str(LOCAL_ROOT))   # script uses relative paths from cwd

# Override DATA_DIR / OUT_DIR inside the script by rewriting the module constants
# We do this cleanly by importing after adjusting sys.path + overriding at module level.

# First, remove any cached version
for mod_name in list(_sys.modules.keys()):
    if "preprocess_data" in mod_name:
        del _sys.modules[mod_name]

# Load the module
import importlib.util
spec = importlib.util.spec_from_file_location(
    "preprocess_data",
    str(LOCAL_ROOT / "scripts" / "preprocess_data.py"),
)
preprocess_data = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preprocess_data)

# Override paths to use Drive for output
preprocess_data.DATA_DIR = DRIVE_DATA
preprocess_data.OUT_DIR  = DRIVE_PROC

# ── Check if processed data already exists on Drive ───────────────────────────
existing_parquets = list(DRIVE_PROC.rglob("*.parquet"))
if existing_parquets and not FORCE_REPROCESS:
    print(f"✓ Found {len(existing_parquets)} existing Parquet files on Drive.")
    print("  Set FORCE_REPROCESS = True to regenerate.\n")
    for f in sorted(existing_parquets)[:12]:
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(DRIVE_PROC)}  ({size_mb:.1f} MB)")
    if len(existing_parquets) > 12:
        print(f"  … and {len(existing_parquets)-12} more")
else:
    import logging, sys
    logging.getLogger().setLevel(logging.INFO)
    if not logging.getLogger().handlers:
        logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))
        
    print("Running full preprocessing pipeline …")
    print("(This processes ~16M+ rows — expect ~10-15 min on Colab L4 instance)")
    preprocess_data.main()
    print("\n✓ Preprocessing complete.")

In [ ]:
# ── Load processed Parquet files and build dataset dicts ─────────────────────
# Expected layout:  DRIVE_PROC/{split}/{PAIR}_{tf}.parquet

def load_split(split: str, pair: str, tf: str) -> pd.DataFrame:
    path = DRIVE_PROC / split / f"{pair}_{tf}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Parquet not found: {path}")
    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    return df

# Quality check — show bar counts and feature columns for GBPJPY 15m train
sample_df = load_split("train", "GBPJPY", "15m")
print("GBPJPY 15m train — shape :", sample_df.shape)
print("Columns :", list(sample_df.columns))
print()
print(sample_df.head(3).to_string())
print()

# Spread check across pairs
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, pair in zip(axes.flat, PAIRS):
    try:
        df_pair = load_split("train", pair, "15m")
        ax.hist(df_pair["spread_avg"].clip(0, 30), bins=60,
                color="#FF9800", edgecolor="none", alpha=0.85)
        ax.set_title(f"{pair} 15m spread (pips)")
        ax.set_xlabel("pips")
        ax.axvline(df_pair["spread_avg"].median(), color="red",
                   linestyle="--", label=f"median={df_pair['spread_avg'].median():.2f}")
        ax.legend(fontsize=8)
    except FileNotFoundError as e:
        ax.text(0.5, 0.5, str(e), ha="center", va="center", transform=ax.transAxes)
plt.suptitle("Spread distributions across pairs (15m train set)", y=1.01)
plt.tight_layout()
plt.show()

## 6. Define the Model Architecture

Import `FluxSignal` (single-TF) and `WaveTraderMTF` (multi-TF) from the cloned repo and print full parameter counts.

In [ ]:
import wavetrader as wt
from wavetrader.model import FluxSignal, WaveTraderMTF
from wavetrader.config import SignalConfig, MTFConfig

# ── Single-TF model ───────────────────────────────────────────────────────────
single_cfg = SignalConfig(
    pair      = "GBP/JPY",
    timeframe = "15min",
    epochs    = 50,
    lookback  = 100,
    batch_size = 64,
    dropout   = 0.2,
    learning_rate = 1e-4,
)
flux_model = FluxSignal(single_cfg)
n_single   = flux_model.count_parameters()

# ── Multi-TF model ────────────────────────────────────────────────────────────
mtf_cfg = MTFConfig(
    pair      = "GBP/JPY",
    epochs    = 50,
    batch_size = 32,
    dropout   = 0.2,
    learning_rate = 1e-4,
)
mtf_model = WaveTraderMTF(mtf_cfg)
n_mtf     = mtf_model.count_parameters()

print("=" * 60)
print("FluxSignal  (single-TF 15m)")
print(f"  Parameters : {n_single:,}")
print(f"  Timeframe  : {single_cfg.timeframe}")
print(f"  Lookback   : {single_cfg.lookback} bars")
print(f"  Input dim  : {single_cfg.total_input_dim}")
print(f"  Output dim : {single_cfg.output_wave_dim}")
print()
print("WaveTraderMTF  (multi-TF 15m/1h/4h/1d)")
print(f"  Parameters : {n_mtf:,}")
print(f"  Timeframes : {mtf_cfg.timeframes}")
print("=" * 60)

# ── Sanity: single forward pass with random data ──────────────────────────────
B, T = 2, single_cfg.lookback
dummy = {
    "ohlcv":     torch.randn(B, T, 5),
    "structure": torch.randn(B, T, 8),
    "rsi":       torch.randn(B, T, 3),
    "volume":    torch.randn(B, T, 3),
    "regime":    torch.randn(B, T, 4),
}
flux_model.eval()
with torch.no_grad():
    out = flux_model(dummy)
print("\nFluxSignal dummy forward pass:")
print(f"  signal_logits : {out['signal_logits'].shape}  (BUY / SELL / HOLD)")
print(f"  confidence    : {out['confidence'].shape}")
print(f"  risk_params   : {out['risk_params'].shape}  (SL / TP / trailing)")
print("\n✓ Architecture OK")

## 7. Configure Training Parameters

Edit the cells below to tune hyperparameters. Defaults are optimised for Colab T4 (16 GB VRAM).

In [ ]:
import torch.utils.data as tud
from wavetrader.dataset import ForexDataset, MTFForexDataset, mtf_collate_fn

# ── Which pair to train on ────────────────────────────────────────────────────
TRAIN_PAIR = "GBP/JPY"      # change to "USD/JPY" | "EUR/JPY" | "GBP/USD"
TRAIN_MODE = "single"       # "single" = FluxSignal  |  "mtf" = WaveTraderMTF

pair_tag   = TRAIN_PAIR.replace("/", "")

# ── Dataset loading helper ────────────────────────────────────────────────────
def _load_parquet(split: str, tf: str) -> pd.DataFrame:
    path = DRIVE_PROC / split / f"{pair_tag}_{tf}.parquet"
    df   = pd.read_parquet(path)
    # ForexDataset expects columns: date, open, high, low, close, volume
    df.index.name = "date"
    df = df.reset_index()
    df = df.rename(columns={
        "open_mid":  "open",
        "high_mid":  "high",
        "low_mid":   "low",
        "close_mid": "close",
    })
    return df

if TRAIN_MODE == "single":
    train_df = _load_parquet("train", "15m")
    val_df   = _load_parquet("val",   "15m")
    test_df  = _load_parquet("test",  "15m")

    print(f"Single-TF datasets  ({TRAIN_PAIR}, 15m)")
    print(f"  Train : {len(train_df):,} bars  "
          f"{train_df['date'].min().date()} → {train_df['date'].max().date()}")
    print(f"  Val   : {len(val_df):,} bars  "
          f"{val_df['date'].min().date()} → {val_df['date'].max().date()}")
    print(f"  Test  : {len(test_df):,} bars  "
          f"{test_df['date'].min().date()} → {test_df['date'].max().date()}")

    # Update config with confirmed sizes
    single_cfg = SignalConfig(
        pair=TRAIN_PAIR, timeframe="15min",
        epochs=50, lookback=100, batch_size=64,
        dropout=0.2, learning_rate=1e-4,
    )
    train_ds = ForexDataset(train_df, single_cfg.lookback, pair=TRAIN_PAIR)
    val_ds   = ForexDataset(val_df,   single_cfg.lookback, pair=TRAIN_PAIR)
    test_ds  = ForexDataset(test_df,  single_cfg.lookback, pair=TRAIN_PAIR)

    train_loader = tud.DataLoader(train_ds, batch_size=single_cfg.batch_size,
                                  shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = tud.DataLoader(val_ds,   batch_size=single_cfg.batch_size,
                                  num_workers=2, pin_memory=True)
    test_loader  = tud.DataLoader(test_ds,  batch_size=single_cfg.batch_size,
                                  num_workers=2, pin_memory=True)

    print(f"\n  DataLoader batches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")
    model_to_train = FluxSignal(single_cfg)

else:  # mtf
    TFS = ["15m", "1h", "4h", "1d"]
    TF_KEY_MAP = {"15m": "15min", "1h": "1h", "4h": "4h", "1d": "1d"}

    def _load_mtf_split(split: str) -> dict:
        out = {}
        for tf in TFS:
            df = _load_parquet(split, tf)
            out[TF_KEY_MAP[tf]] = df
        return out

    train_mtf = _load_mtf_split("train")
    val_mtf   = _load_mtf_split("val")
    test_mtf  = _load_mtf_split("test")

    mtf_cfg = MTFConfig(pair=TRAIN_PAIR, epochs=50, batch_size=32,
                        dropout=0.2, learning_rate=1e-4)
    train_ds = MTFForexDataset(train_mtf, mtf_cfg, pair=TRAIN_PAIR)
    val_ds   = MTFForexDataset(val_mtf,   mtf_cfg, pair=TRAIN_PAIR)
    test_ds  = MTFForexDataset(test_mtf,  mtf_cfg, pair=TRAIN_PAIR)

    train_loader = tud.DataLoader(train_ds, batch_size=mtf_cfg.batch_size,
                                  shuffle=True,  collate_fn=mtf_collate_fn,
                                  num_workers=2, pin_memory=True)
    val_loader   = tud.DataLoader(val_ds,   batch_size=mtf_cfg.batch_size,
                                  collate_fn=mtf_collate_fn,
                                  num_workers=2, pin_memory=True)
    test_loader  = tud.DataLoader(test_ds,  batch_size=mtf_cfg.batch_size,
                                  collate_fn=mtf_collate_fn,
                                  num_workers=2, pin_memory=True)

    print(f"Multi-TF datasets  ({TRAIN_PAIR})")
    for tf, split_name, ds in [("train", train_mtf, train_ds),
                                ("val",   val_mtf,   val_ds),
                                ("test",  test_mtf,  test_ds)]:
        ref = split_name["15min"]
        print(f"  {tf:5s}: {len(ds):,} windows")
    model_to_train = WaveTraderMTF(mtf_cfg)

print(f"\nModel parameters: {model_to_train.count_parameters():,}")
cfg_used = single_cfg if TRAIN_MODE == "single" else mtf_cfg
print(f"Epochs   : {cfg_used.epochs}")
print(f"Batch    : {cfg_used.batch_size}")
print(f"LR       : {cfg_used.learning_rate}")

## 8. Train the Model

In [ ]:
from wavetrader.training import train_model, train_mtf_model, SignalLoss, walk_forward_splits
import torch.nn as nn, torch.nn.functional as F

CKPT_LOCAL = str(LOCAL_ROOT / f"wavetrader_{TRAIN_MODE}_best.pt")

print(f"Training on: {device}")
print(f"Checkpoint will be saved to (local): {CKPT_LOCAL}")
print()

if TRAIN_MODE == "single":
    history = train_model(
        model_to_train, train_loader, val_loader,
        single_cfg, device, checkpoint=CKPT_LOCAL,
    )
else:
    history = train_mtf_model(
        model_to_train, train_loader, val_loader,
        mtf_cfg, device, checkpoint=CKPT_LOCAL,
    )

# ── Training curves ───────────────────────────────────────────────────────────
epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(epochs_ran, history["train_loss"], label="train loss", color="#2196F3")
ax1.plot(epochs_ran, history["val_loss"],   label="val loss",   color="#FF5722")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, [v * 100 for v in history["val_accuracy"]],
         color="#4CAF50", label="val accuracy %")
ax2.axhline(33.3, linestyle="--", color="gray", alpha=0.6, label="random baseline (33%)")
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("%")
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle(f"WaveTrader {TRAIN_MODE.upper()} — {TRAIN_PAIR}", y=1.02)
plt.tight_layout()
plt.show()

best_acc = max(history["val_accuracy"])
print(f"\nBest validation accuracy: {best_acc:.2%}  (baseline random = 33.3%)")

## 9. Save the Model to Google Drive

In [ ]:
import shutil, json, hashlib

RUN_TIMESTAMP = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
SAVE_NAME     = f"wavetrader_{TRAIN_MODE}_{pair_tag}_{RUN_TIMESTAMP}"
DRIVE_SAVE_DIR = DRIVE_CKPT / SAVE_NAME
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Copy the best checkpoint weights
drive_weights = DRIVE_SAVE_DIR / "model_weights.pt"
shutil.copy(CKPT_LOCAL, str(drive_weights))

# 2. Save model config as JSON
cfg_dict = vars(cfg_used).copy()
cfg_dict["_run"] = RUN_TIMESTAMP
cfg_dict["_pair"] = TRAIN_PAIR
cfg_dict["_mode"] = TRAIN_MODE
cfg_dict["_best_val_acc"] = float(max(history["val_accuracy"]))
cfg_dict["_arch_class"] = type(model_to_train).__name__

with open(DRIVE_SAVE_DIR / "config.json", "w") as f:
    json.dump(cfg_dict, f, indent=2, default=str)

# 3. Save training history
with open(DRIVE_SAVE_DIR / "history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.items()}, f, indent=2)

# 4. Compute SHA-256 of saved weights for integrity verification
def _sha256(path: pathlib.Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

weights_hash = _sha256(drive_weights)
with open(DRIVE_SAVE_DIR / "weights.sha256", "w") as f:
    f.write(f"{weights_hash}  model_weights.pt\n")

# 5. Verify
print("✓ Saved to Google Drive:")
for item in sorted(DRIVE_SAVE_DIR.iterdir()):
    size_kb = item.stat().st_size / 1024
    print(f"  {item.name:<35} {size_kb:>8.1f} KB")
print()
print(f"SHA-256 (weights): {weights_hash}")
print(f"Best val accuracy: {cfg_dict['_best_val_acc']:.2%}")
print(f"\nFull path: {DRIVE_SAVE_DIR}")

## 10. Load the Model from Google Drive

Reload into a **fresh** model instance to prove the checkpoint is self-contained. We verify weight integrity via SHA-256 and confirm layer outputs match the training instance.

In [ ]:
# ── Locate the latest checkpoint on Drive ────────────────────────────────────
# (either the one we just saved, or any prior run)
all_run_dirs = sorted(DRIVE_CKPT.glob(f"wavetrader_{TRAIN_MODE}_{pair_tag}_*"),
                      key=lambda p: p.name, reverse=True)
if not all_run_dirs:
    raise RuntimeError(f"No checkpoint dirs found in {DRIVE_CKPT}")

LOAD_DIR = all_run_dirs[0]   # most recent
print(f"Loading from: {LOAD_DIR.name}")

# ── 1. Verify SHA-256 ─────────────────────────────────────────────────────────
sha_file    = LOAD_DIR / "weights.sha256"
weights_file = LOAD_DIR / "model_weights.pt"
expected_hash = sha_file.read_text().split()[0]
actual_hash   = _sha256(weights_file)

if expected_hash == actual_hash:
    print(f"✓ SHA-256 verified: {actual_hash[:16]}…")
else:
    raise RuntimeError(
        f"Checksum MISMATCH!\n  expected: {expected_hash}\n  actual:   {actual_hash}"
    )

# ── 2. Reconstruct config from JSON ──────────────────────────────────────────
with open(LOAD_DIR / "config.json") as f:
    ckpt_cfg_dict = json.load(f)

arch_name = ckpt_cfg_dict.get("_arch_class", "FluxSignal")
print(f"Architecture : {arch_name}")

if arch_name == "FluxSignal":
    loaded_cfg   = SignalConfig(**{k: v for k, v in ckpt_cfg_dict.items()
                                    if not k.startswith("_")
                                    and k in SignalConfig.__dataclass_fields__})
    loaded_model = FluxSignal(loaded_cfg)
else:
    loaded_cfg   = MTFConfig(**{k: v for k, v in ckpt_cfg_dict.items()
                                 if not k.startswith("_")
                                 and k in MTFConfig.__dataclass_fields__})
    loaded_model = WaveTraderMTF(loaded_cfg)

# ── 3. Load weights ───────────────────────────────────────────────────────────
state_dict = torch.load(str(weights_file), map_location=device, weights_only=True)
loaded_model.load_state_dict(state_dict)
loaded_model = loaded_model.to(device)
loaded_model.eval()
print(f"✓ Weights loaded into fresh {arch_name} instance")

# ── 4. Cross-check: compare outputs of trained model vs reloaded model ─────────
model_to_train.eval()
model_to_train = model_to_train.to(device)

test_batch_iter = iter(test_loader)
sample_batch = next(test_batch_iter)
sample_batch = {k: v.to(device) for k, v in sample_batch.items() if isinstance(v, torch.Tensor)}
labels_check = sample_batch.pop("label", None)

with torch.no_grad():
    out_orig   = model_to_train(sample_batch)
    out_loaded = loaded_model(sample_batch)

max_diff = (out_orig["signal_logits"] - out_loaded["signal_logits"]).abs().max().item()
print(f"\nOutput cross-check (signal_logits max diff): {max_diff:.2e}")
if max_diff < 1e-5:
    print("✓ Loaded model outputs IDENTICAL to training instance")
else:
    print("⚠ Small numerical difference (may be OK if < 1e-4)")

## 11. Run Validation and Benchmark Tests

Full evaluation on the **test holdout** (2024-07-01 → 2026-04-04, never seen during training):
- Classification metrics: accuracy, per-class precision/recall/F1
- Confusion matrix
- Confidence calibration curve
- Backtest: bar-by-bar simulation with position sizing, SL/TP, circuit breakers

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# ── Collect predictions over full test set ────────────────────────────────────
loaded_model.eval()
all_preds, all_labels, all_confs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        batch_gpu = {k: v.to(device) for k, v in batch.items()
                     if isinstance(v, torch.Tensor)}
        labels = batch_gpu.pop("label")
        out    = loaded_model(batch_gpu)
        preds  = out["signal_logits"].argmax(-1)
        confs  = out["confidence"].squeeze(-1)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_confs.append(confs.cpu())

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_confs  = torch.cat(all_confs).numpy()

CLASS_NAMES = ["BUY", "SELL", "HOLD"]
accuracy = (all_preds == all_labels).mean() * 100

# ── 1. Classification report ──────────────────────────────────────────────────
print("=" * 60)
print(f"TEST SET RESULTS  ({TRAIN_PAIR}, {TRAIN_MODE.upper()})")
print("=" * 60)
print(f"\nOverall Accuracy : {accuracy:.2f}%  (random baseline = 33.33%)")
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# ── 2. Confusion matrix ───────────────────────────────────────────────────────
cm  = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Confusion Matrix — {TRAIN_PAIR} Test Set")

# ── 3. Confidence distribution per prediction type ────────────────────────────
for i, name in enumerate(CLASS_NAMES):
    mask = all_preds == i
    if mask.sum() > 0:
        axes[1].hist(all_confs[mask], bins=30, alpha=0.6, label=name)
axes[1].set_title("Model Confidence Distribution by Signal")
axes[1].set_xlabel("Confidence score")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── 4. Confidence calibration check ──────────────────────────────────────────
correct_mask = (all_preds == all_labels)
# Bin by confidence
bins     = np.linspace(0, 1, 11)
bin_acc  = []
bin_conf = []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (all_confs >= lo) & (all_confs < hi)
    if m.sum() > 0:
        bin_acc.append(correct_mask[m].mean())
        bin_conf.append(all_confs[m].mean())

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.plot(bin_conf, bin_acc, "o-", color="#2196F3", label="Model")
ax.set_title("Confidence Calibration Curve")
ax.set_xlabel("Mean predicted confidence")
ax.set_ylabel("Actual accuracy")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Live Backtest on test set ──────────────────────────────────────────────
from wavetrader.backtest import run_backtest
from wavetrader.config   import BacktestConfig

bt_config = BacktestConfig(
    initial_balance        = 25_000.0,
    risk_per_trade         = 0.01,       # 1% risk per trade
    min_confidence         = 0.55,       # only trade high-confidence signals
    spread_pips            = 3.0,
    pip_value              = 10.0,       # USD/pip for standard lot GBP/JPY
    leverage               = 30,
    atr_halt_multiplier    = 3.0,
    drawdown_reduce_threshold = 0.10,
)

print("Running backtest on test holdout …")
results = run_backtest(loaded_model, test_df, cfg_used, bt_config, device)

# ── Backtest summary ──────────────────────────────────────────────────────────
roi = (results.final_balance / bt_config.initial_balance - 1) * 100
print()
print("=" * 60)
print(f"  BACKTEST RESULTS  ({TRAIN_PAIR}, test 2024-07-01 → 2026-04-04)")
print("=" * 60)
print(f"  Starting Capital : ${bt_config.initial_balance:>12,.2f}")
print(f"  Ending Capital   : ${results.final_balance:>12,.2f}")
print(f"  Net P&L          : ${results.total_pnl:>12,.2f}")
print(f"  ROI              : {roi:>+.2f}%")
print(f"  Max Drawdown     : {results.max_drawdown:.2%}")
print(f"  Win Rate         : {results.win_rate:.2%}")
print(f"  Profit Factor    : {results.profit_factor:.2f}")
print(f"  Sharpe Ratio     : {results.sharpe_ratio:.2f}")
print(f"  Total Trades     : {results.total_trades}")
print(f"  Winning / Losing : {results.winning_trades} / {results.losing_trades}")
print("=" * 60)

# ── Equity curve ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(results.equity_curve, lw=1.2, color="#4CAF50")
ax.axhline(bt_config.initial_balance, linestyle="--", color="gray", alpha=0.7, label="Start")
ax.fill_between(range(len(results.equity_curve)), bt_config.initial_balance,
                results.equity_curve,
                where=[e > bt_config.initial_balance for e in results.equity_curve],
                color="#4CAF50", alpha=0.15)
ax.fill_between(range(len(results.equity_curve)), bt_config.initial_balance,
                results.equity_curve,
                where=[e < bt_config.initial_balance for e in results.equity_curve],
                color="#F44336", alpha=0.15)
ax.set_title(f"Equity Curve — {TRAIN_PAIR} Test Backtest")
ax.set_xlabel("Bar")
ax.set_ylabel("Balance (USD)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Sample predictions vs ground truth ────────────────────────────────────────
print("\nSample predictions (first 15 test windows):")
print(f"  {'True':>6}  {'Pred':>6}  {'Conf':>6}  {'Match':>6}")
print(f"  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*6}")
for i in range(min(15, len(all_labels))):
    t, p, c = CLASS_NAMES[all_labels[i]], CLASS_NAMES[all_preds[i]], all_confs[i]
    match = "✓" if all_labels[i] == all_preds[i] else "✗"
    print(f"  {t:>6}  {p:>6}  {c:>6.3f}  {match:>6}")

---
## Summary

| Metric | Value |
|--------|-------|
| Model saved to | `MyDrive/phase_lm/checkpoints/wavetrader_{mode}_{pair}_{timestamp}/` |
| Files | `model_weights.pt`, `config.json`, `history.json`, `weights.sha256` |
| Checkpoint verified | SHA-256 hash match + output cross-check < 1e-5 diff |

**To reload in a future session:**
```python
LOAD_DIR = sorted(DRIVE_CKPT.glob("wavetrader_single_GBPJPY_*"))[-1]
cfg      = SignalConfig(**{k: v for k, v in json.load(open(LOAD_DIR/"config.json")).items()
                           if not k.startswith("_") and k in SignalConfig.__dataclass_fields__})
model    = FluxSignal(cfg)
model.load_state_dict(torch.load(LOAD_DIR/"model_weights.pt", weights_only=True))
model.eval()
```